<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/TDConnect4Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install bitbully[gui]

In [1]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

# Export Legacy Format to Standardized format:

In [2]:
from techdays26.legacy_ntuple_agent import TDWeightsLoader

ModuleNotFoundError: No module named 'bitbully.agent_interface'

In [ ]:
if True:
  loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)
  both = loader.load_two_player_from_zip("C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt")

  # Export to a clean single file:
  export_two_player_weights_zip("td_weights_clean.tdw.zip", both)

# Later / elsewhere:
both2 = import_two_player_weights_zip("td_weights_clean.tdw.zip", int_dtype=np.int32)

evaluator = TDEvaluator(both2)
td_agent = TDConnect4Agent(evaluator, tie_break="center")

NameError: name 'import_two_player_weights_zip' is not defined

In [ ]:
# =============================================================================
# Example usage (including Protocol check)
# =============================================================================
if False:
  loader = TDWeightsLoader(int_dtype=np.int32, validate=True, strict_t=True)

  # Directory:
  # both = loader.load_two_player(".")

  # Zip:
  both = loader.load_two_player_from_zip("C4.TCL-EXP-lam0.5-et.txt.zip", p0="p0.txt", p1="p1.txt")

  evaluator = TDEvaluator(both)
  td_agent = TDConnect4Agent(evaluator, tie_break="center")


import bitbully
board = bitbully.Board()
board.play("3334101133114410")

# Structural/runtime Protocol check (optional):
from bitbully.agent_interface import Connect4Agent
assert isinstance(td_agent, Connect4Agent)

print(td_agent.score_all_moves(board))
print("best:", td_agent.best_move(board))
print("score col 3:", td_agent.score_move(board, 3))


In [ ]:
from bitbully import BitBully
from bitbully.agent_interface import Connect4Agent
bitbully_agent = BitBully(opening_book="12-ply-dist", tie_break="random")

In [ ]:
agents: dict[str, Connect4Agent] = {
    "BitBully-12ply": bitbully_agent,
    "TD-Agent": td_agent,
}

In [ ]:
from bitbully.gui_c4 import GuiC4

%matplotlib ipympl

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())

In [ ]:
from __future__ import annotations

from typing import Protocol

def play_match(
    agent_yellow: Connect4Agent,
    agent_red: Connect4Agent,
    *,
    start: bitbully.Board | None = None,
    max_plies: int = 42,
    verbose: int = 0,
) -> int:
    """
    Play a full game between two agents on your BitBully `Board`.

    Returns:
        0 -> draw
        1 -> yellow win
        2 -> red win
    """
    board = start.copy() if start is not None else bitbully.Board()

    # Safety: if someone passes a terminal board
    if board.is_game_over():
        w = board.winner()
        return 0 if w is None else int(w)

    plies = 0
    while not board.is_game_over():
        assert plies < max_plies
        # Should never happen in Connect-4, but keeps us robust

        player = board.current_player()  # 1=yellow, 2=red
        agent = agent_yellow if player == 1 else agent_red

        move = agent.best_move(board)

        if not board.is_legal_move(move):
            raise ValueError(
                f"Agent selected illegal move {move} for player {player}. "
                f"Legal moves: {board.legal_moves()}"
            )

        ok = board.play(move)
        if not ok:
            # `play()` returns False on illegal moves; we already checked, so this would indicate a bug.
            raise RuntimeError(f"Board.play({move}) returned False although move was legal.")

        plies += 1

        if verbose >= 1:
            print(board)

    w = board.winner()
    return 0 if w is None else int(w)

In [ ]:
import bitbully

result = play_match(td_agent, bitbully_agent, verbose=1)
# 0 draw, 1 yellow win, 2 red win
print(result)

In [ ]:
from collections import defaultdict

n_matches = 200
total_scores = defaultdict(int)
for _ in range(n_matches):
    result = play_match(td_agent, bitbully_agent, verbose=0)
    total_scores[result] += 1

In [ ]:
total_scores

In [ ]:
# -----------------------------
# 2) Arena config
# -----------------------------

# Logger is optional; arena is silent by default unless you configure logging.
logger = logging.getLogger("bitbully.arena")
logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout
# If you want to see logs in console:
# logging.basicConfig(level=logging.WARNING)

cfg = ArenaConfig(
    agents=(
        AgentSpec(
            agent_id="random_agent",
            agent=RandomAgent(),
            colors=(Color.YELLOW, Color.RED),  # can play both
            epsilons=(0.00,),
        ),
        AgentSpec(
            agent_id="bitbully_12plydist_random",
            agent=bitbully_agent,
            colors=(Color.YELLOW, Color.RED),  # can play both
            epsilons=(0.00, 0.05, 0.10, 0.15, 0.2, 0.25),
        ),
        AgentSpec(
            agent_id="td_center",
            agent=td_agent,
            colors=(Color.YELLOW, Color.RED),  # can play both,
            epsilons=(0.00,),
        ),
    ),
    n_games=100,  # n games per pairing per ε per color-swap
    time_control=TimeControl(
        per_move_timeout_s=2.0,  # best-effort (measured after return)
        per_game_budget_s=45.0,    # seconds of total think time
    ),
    seed=12345,
    log_scores=False,   # set True to also call score_all_moves() for logging
    use_tqdm=True,      # optional progress bar
    logger=logger,
)

# -----------------------------
# 3) Run arena
# -----------------------------

arena = BitBullyArena()
result = arena.run(cfg)

In [ ]:
# -----------------------------
# 4) Inspect results
# -----------------------------

print("Skipped pairings:", len(result.skipped))
print("Games played:", len(result.games))
print("Aggregate rows:", len(result.aggregates))

# Print aggregate rows (one row per (yellow_agent, red_agent, eps_y, eps_r))
for row in result.aggregates:
    print(row)

# Look at a single game record (includes move list + per-move metadata)
g0 = result.games[0]
print("First game:", g0.game_cfg)
print("Termination:", g0.termination, "winner:", g0.winner)
print("Moves:", g0.moves)
print("First move meta:", g0.move_meta[0])